In [1]:
import sys, time
import numpy as np
import pandas as pd
sys.path.append("../src")
import features
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/cleaned_cars.csv")
X = df[features.RAW_INPUTS]
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
print(X_train.shape, X_test.shape)

(48819, 12) (12205, 12)


In [2]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_validate, KFold

models = {
    "Baseline (median)": DummyRegressor(strategy="median"),
    "Linear (Ridge)": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                                           n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                   random_state=42),
}
cv = KFold(n_splits=3, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    t = time.time()
    res = cross_validate(features.build_pipeline(clone(model)), X_train, y_train_log,
                         cv=cv, n_jobs=-1,
                         scoring=["neg_mean_absolute_error",
                                  "neg_root_mean_squared_error", "r2"])
    rows.append({"model": name,
                 "MAE (log)": -res["test_neg_mean_absolute_error"].mean(),
                 "RMSE (log)": -res["test_neg_root_mean_squared_error"].mean(),
                 "R2": res["test_r2"].mean(),
                 "seconds": time.time() - t})
    print(name, "done")

cv_table = pd.DataFrame(rows).round(3).set_index("model")
cv_table

Baseline (median) done
Linear (Ridge) done
Random Forest done
Gradient Boosting done


,MAE (log),RMSE (log),R2,seconds
model,,,,
Baseline (median),0.671,0.874,-0.001,13.331
Linear (Ridge),0.133,0.201,0.947,7.919
Random Forest,0.101,0.164,0.965,455.307
Gradient Boosting,0.120,0.177,0.959,46.861


### Model comparison (3-fold cross-validation, training set only, log scale)

| Model | MAE (log) | RMSE (log) | R2 |
|---|---|---|---|
| Baseline (median) | 0.671 | 0.874 | -0.001 |
| Linear (Ridge) | 0.133 | 0.201 | 0.947 |
| Random Forest | 0.101 | 0.164 | 0.965 |
| Gradient Boosting | 0.120 | 0.177 | 0.959 |

**Result:** Random Forest has the lowest cross-validated RMSE and beats the median baseline by a wide margin (MAE 0.101 vs 0.671 on the log scale, roughly a 10% typical error). It is selected as the final model. The test set was not used for this choice.

In [3]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

best = models["Random Forest"]

# 1) price range: out-of-fold errors on TRAIN set (log scale)
cv_pred = cross_val_predict(features.build_pipeline(clone(best)),
                            X_train, y_train_log, cv=cv, n_jobs=-1)
q10, q90 = np.quantile(y_train_log - cv_pred, [0.10, 0.90])
print("Range factors:", np.exp(q10), np.exp(q90))

# 2) final fit on full train set, outputs rupees
def make_final(m):
    return TransformedTargetRegressor(regressor=features.build_pipeline(clone(m)),
                                      func=np.log1p, inverse_func=np.expm1)

final = make_final(best).fit(X_train, y_train)
baseline = make_final(models["Baseline (median)"]).fit(X_train, y_train)

# 3) THE one final test evaluation
def report(model):
    p = model.predict(X_test)
    return {"MAE (PKR)": mean_absolute_error(y_test, p),
            "RMSE (PKR)": mean_squared_error(y_test, p) ** 0.5,
            "R2": r2_score(y_test, p),
            "Median abs % error": float(np.median(np.abs(p - y_test) / y_test) * 100)}

test_table = pd.DataFrame({"Baseline (median)": report(baseline),
                           "Random Forest": report(final)}).T.round(3)
test_table

Range factors: 0.8599734914490651 1.1587368447834365


,MAE (PKR),RMSE (PKR),R2,Median abs % error
Baseline (median),2470632.036,6007509.997,-0.049,49.533
Random Forest,376168.058,1591639.825,0.926,6.103


### Final test-set results (evaluated once, after model selection)

| Metric | Baseline (median) | Random Forest |
|---|---|---|
| MAE (PKR) | 2,470,632 | 376,168 |
| RMSE (PKR) | 6,007,510 | 1,591,640 |
| R2 | -0.049 | 0.926 |
| Median abs % error | 49.5% | 6.1% |

The model beats the median baseline by a wide margin. Price range in the app uses out-of-fold CV residuals: low = 0.86 x prediction, high = 1.159 x prediction (about 80% interval).

In [4]:
import joblib, json
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
joblib.dump(final, "../models/pricesense_pipeline.joblib", compress=3)

size_mb = Path("../models/pricesense_pipeline.joblib").stat().st_size / 1e6
print(f"Model size: {size_mb:.1f} MB")

# Feature importance (grouped back to original columns)
inner = final.regressor_
est = inner.named_steps["model"]
names = inner.named_steps["prep"].get_feature_names_out()

def original_column(encoded_name):
    base = encoded_name.split("__", 1)[1]
    for col in sorted(features.NUMERIC + features.CATEGORICAL, key=len, reverse=True):
        if base == col or base.startswith(col + "_"):
            return col
    return base

importance = (pd.Series(est.feature_importances_, index=names)
              .groupby(original_column).sum().sort_values(ascending=False))
importance.to_csv("../models/feature_importance.csv", header=["importance"])

meta = {"residual_q10": float(q10), "residual_q90": float(q90),
        "range_low_factor": float(np.exp(q10)), "range_high_factor": float(np.exp(q90)),
        "test_table": test_table.reset_index().to_dict("records")}
Path("../models/meta.json").write_text(json.dumps(meta, indent=2))

importance.head(10).round(3)

Model size: 59.9 MB


n_features    0.440
car_age       0.217
engine_cc     0.181
brand_tier    0.068
model         0.023
body_type     0.013
make          0.012
fuel          0.010
mileage_km    0.008
assembly      0.008
dtype: float64

### Feature importance (Random Forest, grouped by original feature)

`n_features` (0.44), `car_age` (0.22) and `engine_cc` (0.18) account for about 84% of the importance, followed by `brand_tier` (0.07). `mileage_km` contributes little (0.008), consistent with its weak correlation in the EDA. Note that `n_features` partly proxies for newer, higher-trim cars.